In [1]:
import os
import pandas as pd

# =========================================================
# CONFIGURACIÓN
# =========================================================

BASE_DIR = r"../Events/CSV"   # Carpeta donde están todos los eventos

# Palabras clave para identificar archivos
CATEGORY_MAP = {
    "electron": ["GsfElectrons"],
    "muon": ["TrackerMuons_V2"],
    "photon": ["Photons"],
    "track": ["Tracks_V3"]
}

# Dónde guardaremos el CSV final por cada categoría
OUTPUT = {
    category: []
    for category in CATEGORY_MAP.keys()
}

# =========================================================
# FUNCIÓN DE CARGA SEGURA
# =========================================================

def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Error leyendo {path}: {e}")
        return None

# =========================================================
# RECORRER TODOS LOS EVENTOS
# =========================================================

for event_folder in os.listdir(BASE_DIR):
    full_event_path = os.path.join(BASE_DIR, event_folder)

    if not os.path.isdir(full_event_path):
        continue

    event_id = event_folder  # Nombre de la carpeta = ID del evento

    # Leer todos los CSV del evento
    for fname in os.listdir(full_event_path):
        if not fname.lower().endswith(".csv"):
            continue

        file_path = os.path.join(full_event_path, fname)

        df = safe_read_csv(file_path)
        if df is None or df.empty:
            continue

        # Asignar event_id
        df["event_id"] = event_id

        # Determinar categoría del archivo
        for category, keywords in CATEGORY_MAP.items():
            if any(k.lower() in fname.lower() for k in keywords):
                OUTPUT[category].append(df)
                break

# =========================================================
# GUARDAR CSV MAESTROS
# =========================================================

SAVE_DIR = "CSV_MASTER"
os.makedirs(SAVE_DIR, exist_ok=True)

for category, df_list in OUTPUT.items():
    if df_list:
        merged = pd.concat(df_list, ignore_index=True)
        out_path = os.path.join(SAVE_DIR, f"all_{category}s.csv")
        merged.to_csv(out_path, index=False)
        print(f"[OK] Guardado: {out_path} ({len(merged)} filas)")
    else:
        print(f"[WARN] No hay datos para la categoría: {category}")


[OK] Guardado: CSV_MASTER/all_electrons.csv (146 filas)
[OK] Guardado: CSV_MASTER/all_muons.csv (397 filas)
[OK] Guardado: CSV_MASTER/all_photons.csv (251 filas)
[OK] Guardado: CSV_MASTER/all_tracks.csv (30111 filas)
